# Helper functions - set paths


In [1]:
# --- SETUP: shared paths + helpers (run once) ---
from pathlib import Path
import numpy as np
import nibabel as nib
from functools import lru_cache

SRC_ROOT     = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/A_A_Combined_Data/Processed_HiresLowres_Split_Data/test_hires")
SRC_T1_DIR   = SRC_ROOT / "t1"
SRC_MASK_DIR = SRC_ROOT / "masks"
if not (SRC_T1_DIR.exists() and SRC_MASK_DIR.exists()):
    raise FileNotFoundError("Expected t1/ and masks/ subfolders under test_hires")

OUT_ROOT      = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/A_A_Combined_Data/Downsampled_Data")
OUT_ROOT.mkdir(parents=True, exist_ok=True)
JITTER_OUT     = OUT_ROOT / "test_hires_motion_slicejitter"
JITTER_T1_DIR  = JITTER_OUT / "t1"
JITTER_MASK_DIR= JITTER_OUT / "masks"
for d in (JITTER_OUT, JITTER_T1_DIR, JITTER_MASK_DIR):
    d.mkdir(parents=True, exist_ok=True)

def is_mask_name(name: str) -> bool:
    n = name.lower()
    return ("mask" in n) or ("lesion" in n)

def strip_ext(name: str) -> str:
    return name[:-7] if name.endswith(".nii.gz") else Path(name).stem

def norm_key(name: str) -> str:
    stem = strip_ext(name)
    for suf in ["_T1w_MNI_norm","_T1w_MNI","_T1w_brain","_T1w","_T1","_image","_img","_img_prepped"]:
        if stem.endswith(suf):
            stem = stem[: -len(suf)]
            break
    for suf in ["_lesion_mask_MNI_clean","_lesion_mask_MNI","_lesion_mask","_desc-lesion_mask","_mask","_mask_prepped"]:
        if stem.endswith(suf):
            stem = stem[: -len(suf)]
            break
    return stem.rstrip("_")

def _is_top_level(p: Path) -> bool:
    try:
        parent = p.parent.resolve()
    except Exception:
        return False
    return parent in {SRC_T1_DIR.resolve(), SRC_MASK_DIR.resolve()}

def _pair_maps(img_dir: Path, mask_dir: Path):
    imgs, msks = {}, {}
    for p in img_dir.rglob("*.nii.gz"):
        if is_mask_name(p.name):
            continue
        key = norm_key(p.name)
        keep = imgs.get(key)
        if key and (keep is None or (_is_top_level(p) and not _is_top_level(keep))):
            imgs[key] = p
    for p in mask_dir.rglob("*.nii.gz"):
        if not is_mask_name(p.name):
            continue
        key = norm_key(p.name)
        keep = msks.get(key)
        if key and (keep is None or (_is_top_level(p) and not _is_top_level(keep))):
            msks[key] = p
    keys = sorted(set(imgs) & set(msks))
    return {k: {"img": imgs[k], "msk": msks[k]} for k in keys}

def discover_pairs(img_dir: Path, mask_dir: Path):
    return [(v["img"], v["msk"]) for v in _pair_maps(img_dir, mask_dir).values()]

# basic I/O helpers reused below
@lru_cache(maxsize=128)
def load_nii(path: Path | str) -> nib.Nifti1Image:
    return nib.load(str(path))

def data_f32(img: nib.Nifti1Image) -> np.ndarray:
    return np.asarray(img.get_fdata(dtype=np.float32), dtype=np.float32)

def save_like(ref_img: nib.Nifti1Image, array: np.ndarray, out_path: Path, dtype=None):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    arr = array.astype(dtype or np.float32)
    nib.save(nib.Nifti1Image(arr, ref_img.affine, ref_img.header.copy()), str(out_path))

def pad_or_crop_to(arr: np.ndarray, target_shape: tuple[int, int, int]) -> np.ndarray:
    out = arr
    for axis, tgt in enumerate(target_shape):
        cur = out.shape[axis]
        if cur == tgt:
            continue
        if cur > tgt:
            start = (cur - tgt) // 2
            sl = [slice(None)] * out.ndim
            sl[axis] = slice(start, start + tgt)
            out = out[tuple(sl)]
        else:
            pad_before = (tgt - cur) // 2
            pad_after = tgt - cur - pad_before
            pads = [(0, 0)] * out.ndim
            pads[axis] = (pad_before, pad_after)
            out = np.pad(out, pads, mode="edge")
    return out.astype(arr.dtype, copy=False)

# Preview pairing status up front
img_candidates  = [p for p in SRC_T1_DIR.rglob("*.nii.gz")]
mask_candidates = [p for p in SRC_MASK_DIR.rglob("*.nii.gz")]
_pair_preview   = _pair_maps(SRC_T1_DIR, SRC_MASK_DIR)
print(
    f"Found pairs: {len(_pair_preview)}  (imgs={len(img_candidates)}, masks={len(mask_candidates)})"
)
# (Viewer moved to Cell 7; helpers above are now shared by all cells.)

Found pairs: 214  (imgs=214, masks=214)


# 5) Slice-wise motion (rigid jitter)

Why / real-world: Patient motion during 2D acquisitions → slice-to-slice misalignment/blur.

What the code does

For each slice, applies a small random rotation (±a few degrees) and pixel shift (±a few px):
rotate(..., order=1) for image, order=0 for mask; then shift(...) similarly.

This produces slice-to-slice misalignments and slight blurring/ghosting from interpolation.

Voxel spacing unchanged; just geometry perturbations.

What this mimics

2D multi-slice acquisitions where the patient moves between slice excitations → slice stack doesn’t line up perfectly.

Very common in restless patients, pediatrics, or longer scans.

Why it’s useful

Motion is one of the biggest real-world degraders. Even tiny rotations/shift destroy fine boundaries and create zebra-like slice seams.

Caveats

Real motion can be continuous and within-TR; this is a discrete per-slice model (captures the dominant visual effect).

In [2]:
# === 5) SLICE-WISE MOTION JITTER: small per-slice rotations/shifts (XY plane), Z intact ===
from pathlib import Path
import numpy as np
import nibabel as nib
from scipy.ndimage import rotate, shift
import shutil

OUT_DIR     = JITTER_OUT
OUT_IMG_DIR = JITTER_T1_DIR
OUT_MSK_DIR = JITTER_MASK_DIR
deg_range  = 2.0
px_range   = 2.0
SEED       = 7
OVERWRITE  = False

OUT_DIR.mkdir(parents=True, exist_ok=True)

pairs = discover_pairs(SRC_T1_DIR, SRC_MASK_DIR)
print(f"Discovered unique pairs: {len(pairs)} (deduped across flat + subfolders)")

rng   = np.random.default_rng(SEED)
wrote = 0

for i, (img_p, msk_p) in enumerate(pairs, 1):
    img_ref = load_nii(img_p)
    x = data_f32(img_ref)
    H, W, Z = x.shape
    xm = np.empty_like(x, dtype=np.float32)

    angs = rng.uniform(-deg_range, deg_range, size=Z)
    dxs  = rng.uniform(-px_range,  px_range,  size=Z)
    dys  = rng.uniform(-px_range,  px_range,  size=Z)

    for k in range(Z):
        ang, dx, dy = float(angs[k]), float(dxs[k]), float(dys[k])
        sl = rotate(x[:, :, k], angle=ang, reshape=False, order=1, mode="nearest")
        sl = shift (sl,        shift=(dy, dx),      order=1, mode="nearest")
        xm[:, :, k] = sl

    base = strip_ext(img_p.name).replace("_T1w_MNI_norm","").replace("_T1w","")
    out_img = OUT_IMG_DIR / f"{base}_T1w_MNI_norm.nii.gz"
    out_msk = OUT_MSK_DIR / f"{base}_lesion_mask_MNI_clean.nii.gz"

    if not OVERWRITE and out_img.exists() and out_msk.exists():
        if i % 25 == 0 or i == len(pairs):
            print(f"[{i}/{len(pairs)}] (skip, exists) {out_img.name}")
        continue

    save_like(img_ref, xm, out_img, dtype=np.float32)
    out_msk.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(str(msk_p), str(out_msk))
    wrote += 1

    if i % 10 == 0 or i == len(pairs):
        print(f"[{i}/{len(pairs)}] wrote {out_img.name} & {out_msk.name}")

print(f"Done → {OUT_DIR} | wrote {wrote} case(s)")


Discovered unique pairs: 214 (deduped across flat + subfolders)
[10/214] wrote sub-M2074_ses-2183_T1w_MNI_norm.nii.gz & sub-M2074_ses-2183_lesion_mask_MNI_clean.nii.gz
[10/214] wrote sub-M2074_ses-2183_T1w_MNI_norm.nii.gz & sub-M2074_ses-2183_lesion_mask_MNI_clean.nii.gz
[20/214] wrote sub-M2100_ses-6369_T1w_MNI_norm.nii.gz & sub-M2100_ses-6369_lesion_mask_MNI_clean.nii.gz
[20/214] wrote sub-M2100_ses-6369_T1w_MNI_norm.nii.gz & sub-M2100_ses-6369_lesion_mask_MNI_clean.nii.gz
[30/214] wrote sub-M2140_ses-631_T1w_MNI_norm.nii.gz & sub-M2140_ses-631_lesion_mask_MNI_clean.nii.gz
[30/214] wrote sub-M2140_ses-631_T1w_MNI_norm.nii.gz & sub-M2140_ses-631_lesion_mask_MNI_clean.nii.gz
[40/214] wrote sub-M2175_ses-703_T1w_MNI_norm.nii.gz & sub-M2175_ses-703_lesion_mask_MNI_clean.nii.gz
[40/214] wrote sub-M2175_ses-703_T1w_MNI_norm.nii.gz & sub-M2175_ses-703_lesion_mask_MNI_clean.nii.gz
[50/214] wrote sub-M2220_ses-359_T1w_MNI_norm.nii.gz & sub-M2220_ses-359_lesion_mask_MNI_clean.nii.gz
[50/214] w

In [ ]:
import numpy as np, nibabel as nib, matplotlib.pyplot as plt, ipywidgets as W
from functools import lru_cache
from IPython.display import display, clear_output

orig_pairs = _pair_maps(SRC_T1_DIR, SRC_MASK_DIR)
ds_pairs   = _pair_maps(JITTER_T1_DIR, JITTER_MASK_DIR)
common_keys = sorted(set(orig_pairs) & set(ds_pairs))
if not common_keys:
    raise RuntimeError("No overlapping cases between original and motion-jitter datasets.")

@lru_cache(maxsize=64)
def _load_vol(path: str):
    img = nib.load(path)
    data = img.get_fdata().astype(np.float32)
    return data[..., 0] if data.ndim == 4 and data.shape[-1] == 1 else data

def _normalize(vol):
    nz = vol[vol > 0]
    if nz.size == 0:
        return vol * 0
    p1, p99 = np.percentile(nz, [1, 99])
    return np.clip((vol - p1) / max(p99 - p1, 1e-5), 0, 1)

key_dd   = W.Dropdown(options=common_keys, description="Case:", layout=W.Layout(width="45%"))
slice_sl = W.IntSlider(description="Slice:", min=0, max=1, value=0, continuous_update=False, layout=W.Layout(width="45%"))
out = W.Output()

def _update_slider(*_):
    vol = _load_vol(str(orig_pairs[key_dd.value]["img"]))
    slice_sl.max = max(0, vol.shape[2] - 1)

def _render(*_):
    with out:
        clear_output(wait=True)
        key = key_dd.value
        img_orig = _load_vol(str(orig_pairs[key]["img"]))
        msk_orig = (_load_vol(str(orig_pairs[key]["msk"])) > 0.5)
        img_ds   = _load_vol(str(ds_pairs[key]["img"]))
        msk_ds   = (_load_vol(str(ds_pairs[key]["msk"])) > 0.5)
        z = int(slice_sl.value)

        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        titles = ["Original", "Slice-wise motion jitter"]
        for ax, img, msk, title in zip(axes,
                                       (_normalize(img_orig[..., z]), _normalize(img_ds[..., z])),
                                       (msk_orig[..., z], msk_ds[..., z]),
                                       titles):
            ax.imshow(img.T, cmap="gray", origin="lower")
            ax.contour(msk.T, levels=[0.5], colors="r", linewidths=0.8)
            ax.set_title(f"{title}\n{key} | slice {z}")
            ax.axis("off")
        plt.tight_layout(); plt.show()

_update_slider()
_render()
key_dd.observe(_update_slider, names="value")
key_dd.observe(_render, names="value")
slice_sl.observe(_render, names="value")
display(W.VBox([W.HBox([key_dd, slice_sl]), out]))

In [ ]:
# === Comparison viewer: HiRes → Rigid Jitter ===
# Self-contained — run without executing other cells.
# Panel 4: PERILESIONAL CONTRAST = voxel intensity − mean(5 mm surrounding ring)

from pathlib import Path
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy.ndimage import binary_dilation

HIRES_T1_DIR   = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/A_A_Combined_Data/Processed_HiresLowres_Split_Data/test_hires/t1")
HIRES_MASK_DIR = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/A_A_Combined_Data/Processed_HiresLowres_Split_Data/test_hires/masks")
DEG_T1_DIR     = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/A_A_Combined_Data/Downsampled_Data/RigidJitter/t1")
METHOD_NAME    = "Rigid Jitter (slice-wise motion)"
SAVE_PATH      = DEG_T1_DIR.parent / "comparison_viewer.png"
DILATION_ITERS = 5
COLOR_HI       = "#3a86ff"
COLOR_DEG      = "#ff6b35"

def best_axial_slice(mask):
    return int(np.argmax(mask.sum(axis=(0, 1))))

def brain_bbox(sl, pad=12):
    rows = np.where(sl.max(axis=1) > 0)[0]
    cols = np.where(sl.max(axis=0) > 0)[0]
    if len(rows) == 0 or len(cols) == 0:
        return slice(None), slice(None)
    return (slice(max(rows[0] - pad, 0), min(rows[-1] + pad + 1, sl.shape[0])),
            slice(max(cols[0] - pad, 0), min(cols[-1] + pad + 1, sl.shape[1])))

def perilesional_contrast_voxels(vol, mask):
    dilated   = binary_dilation(mask, iterations=DILATION_ITERS)
    peri      = dilated & ~mask
    peri_mean = float(vol[peri].mean()) if peri.sum() > 0 else 0.0
    return vol[mask] - peri_mean

records = []
for deg_path in sorted(DEG_T1_DIR.glob("*_T1w_MNI_norm.nii.gz")):
    key       = deg_path.name.replace("_T1w_MNI_norm.nii.gz", "")
    hi_path   = HIRES_T1_DIR / deg_path.name
    mask_path = HIRES_MASK_DIR / f"{key}_lesion_mask_MNI_clean.nii.gz"
    if hi_path.exists() and mask_path.exists():
        m = nib.load(str(mask_path)).get_fdata() > 0
        if m.sum() > 10:
            records.append((int(m.sum()), key, hi_path, deg_path, mask_path))

records.sort(key=lambda r: r[0])
n_show = 6
chosen = [records[i] for i in np.linspace(0, len(records) - 1, n_show, dtype=int)]
print(f"Matched pairs: {len(records)}  |  Showing {n_show} spanning small → large lesion")

col_w = [1.0, 1.0, 1.0, 1.6]
fig   = plt.figure(figsize=(20, n_show * 3.6))
outer = gridspec.GridSpec(n_show, 4, figure=fig,
                           width_ratios=col_w, wspace=0.10, hspace=0.44)
col_titles = ["HiRes  (original)", METHOD_NAME,
              "Difference  (degraded − orig)", "Local contrast  =  voxel − perilesional mean"]

for row, (nvox, key, hi_path, deg_path, mask_path) in enumerate(chosen):
    hi   = nib.load(str(hi_path)).get_fdata(dtype=np.float32)
    deg  = nib.load(str(deg_path)).get_fdata(dtype=np.float32)
    mask = nib.load(str(mask_path)).get_fdata() > 0

    z       = best_axial_slice(mask)
    hi_sl   = hi[:, :, z].T;   deg_sl  = deg[:, :, z].T
    mask_sl = mask[:, :, z].T; diff_sl = (deg[:, :, z] - hi[:, :, z]).T
    contour = binary_dilation(mask_sl) ^ mask_sl

    rs, cs = brain_bbox(hi_sl)
    hi_c   = hi_sl[rs, cs];   deg_c  = deg_sl[rs, cs]
    diff_c = diff_sl[rs, cs]; msk_c  = mask_sl[rs, cs];  cnt_c = contour[rs, cs]
    dmax   = max(float(np.abs(diff_c).max()), 0.05)

    def add_overlay(ax, base, cnt, msk):
        ax.imshow(base, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
        ov = np.zeros((*base.shape, 4), dtype=float);  ov[cnt] = [0.0, 1.0, 0.3, 0.9]
        ax.imshow(ov, interpolation="nearest")
        fl = np.zeros((*base.shape, 4), dtype=float);  fl[msk] = [1.0, 0.9, 0.0, 0.15]
        ax.imshow(fl, interpolation="nearest");  ax.axis("off")

    ax0 = fig.add_subplot(outer[row, 0]);  add_overlay(ax0, hi_c,  cnt_c, msk_c)
    ax1 = fig.add_subplot(outer[row, 1]);  add_overlay(ax1, deg_c, cnt_c, msk_c)
    ax2 = fig.add_subplot(outer[row, 2])
    im  = ax2.imshow(diff_c, cmap="RdBu_r", vmin=-dmax, vmax=dmax, interpolation="nearest")
    ov2 = np.zeros((*diff_c.shape, 4), dtype=float);  ov2[cnt_c] = [0.0, 1.0, 0.3, 0.9]
    ax2.imshow(ov2, interpolation="nearest");  ax2.axis("off")
    div = make_axes_locatable(ax2)
    fig.colorbar(im, cax=div.append_axes("right", size="6%", pad=0.05)).ax.tick_params(labelsize=6)

    ax3      = fig.add_subplot(outer[row, 3])
    hi_cntr  = perilesional_contrast_voxels(hi,  mask)
    deg_cntr = perilesional_contrast_voxels(deg, mask)
    cmin = min(hi_cntr.min(), deg_cntr.min());  cmax = max(hi_cntr.max(), deg_cntr.max())
    pad  = (cmax - cmin) * 0.06
    bins = np.linspace(cmin - pad, cmax + pad, 50)
    ax3.hist(hi_cntr,  bins=bins, alpha=0.68, color=COLOR_HI,  density=True,
             label=f"HiRes  μ = {hi_cntr.mean():+.3f}", zorder=3)
    ax3.hist(deg_cntr, bins=bins, alpha=0.68, color=COLOR_DEG, density=True,
             label=f"Degraded  μ = {deg_cntr.mean():+.3f}", zorder=3)
    ax3.axvline(0, color="black", lw=1.8, ls="--", zorder=5, label="zero  (same as surroundings)")
    ax3.axvspan(cmin - pad, 0, alpha=0.07, color=COLOR_HI, zorder=0)
    ax3.axvspan(0, cmax + pad, alpha=0.07, color=COLOR_DEG, zorder=0)
    ax3.text(0.02, 0.96, "← darker", transform=ax3.transAxes,
             ha="left", va="top", fontsize=8, color=COLOR_HI, fontweight="bold")
    ax3.text(0.98, 0.96, "brighter →", transform=ax3.transAxes,
             ha="right", va="top", fontsize=8, color=COLOR_DEG, fontweight="bold")
    ax3.set_xlabel("Contrast  (a.u.)", fontsize=9);  ax3.set_ylabel("Density", fontsize=9)
    ax3.tick_params(labelsize=8)
    ax3.legend(fontsize=8, loc="upper center", framealpha=0.88)
    ax3.spines["top"].set_visible(False);  ax3.spines["right"].set_visible(False)

    if row == 0:
        for ax, title in zip([ax0, ax1, ax2, ax3], col_titles):
            ax.set_title(title, fontsize=11, fontweight="bold", pad=8)
    ax0.set_ylabel(f"{key[-20:]}\n{nvox/1000:.1f} mL",
                   fontsize=7.5, rotation=0, labelpad=96, va="center")

fig.suptitle(
    f"Degradation comparison: HiRes → {METHOD_NAME}\n"
    "Lesion contour in green  |  Diff: red = degraded brighter, blue = degraded darker\n"
    "Histogram: local contrast relative to 5 mm perilesional ring",
    fontsize=12, fontweight="bold", y=1.01)
plt.savefig(SAVE_PATH, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {SAVE_PATH}")

**Figure 2. Population-level perilesional contrast: HiRes vs Rigid Jitter (n = 138).** *(Left)* Scatter plot of lesion perilesional contrast in HiRes (x-axis) versus Rigid-Jitter-degraded (y-axis) images, coloured by log-transformed lesion volume. Slice-wise motion creates partial-volume mixing at lesion boundaries; this may broaden the degraded distribution and shift some subjects toward zero contrast. *(Right)* Overlapping population histograms show whether slice-wise jitter attenuates the perilesional contrast signal relative to HiRes baseline.

In [ ]:
# === Summary: perilesional contrast HiRes vs Rigid Jitter — all subjects ===
# Self-contained — run without executing other cells.

from pathlib import Path
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from scipy.ndimage import binary_dilation

HIRES_T1_DIR   = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/A_A_Combined_Data/Processed_HiresLowres_Split_Data/test_hires/t1")
HIRES_MASK_DIR = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/A_A_Combined_Data/Processed_HiresLowres_Split_Data/test_hires/masks")
DEG_T1_DIR     = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/A_A_Combined_Data/Downsampled_Data/RigidJitter/t1")
METHOD_NAME    = "Rigid Jitter (slice-wise motion)"
SAVE_PATH      = DEG_T1_DIR.parent / "perilesional_contrast_summary.png"
DILATION_ITERS = 5
COLOR_HI       = "#3a86ff"
COLOR_DEG      = "#ff6b35"

def peri_contrast_mean(vol, mask):
    dilated   = binary_dilation(mask, iterations=DILATION_ITERS)
    peri      = dilated & ~mask
    peri_mean = float(vol[peri].mean()) if peri.sum() > 0 else 0.0
    return float(vol[mask].mean()) - peri_mean

hi_c, deg_c, nvoxs = [], [], []
for deg_path in sorted(DEG_T1_DIR.glob("*_T1w_MNI_norm.nii.gz")):
    key       = deg_path.name.replace("_T1w_MNI_norm.nii.gz", "")
    hi_path   = HIRES_T1_DIR / deg_path.name
    mask_path = HIRES_MASK_DIR / f"{key}_lesion_mask_MNI_clean.nii.gz"
    if not (hi_path.exists() and mask_path.exists()):
        continue
    hi   = nib.load(str(hi_path)).get_fdata(dtype=np.float32)
    deg  = nib.load(str(deg_path)).get_fdata(dtype=np.float32)
    mask = nib.load(str(mask_path)).get_fdata() > 0
    if mask.sum() < 10:
        continue
    hi_c.append(peri_contrast_mean(hi,  mask))
    deg_c.append(peri_contrast_mean(deg, mask))
    nvoxs.append(int(mask.sum()))

hi_c  = np.array(hi_c);   deg_c = np.array(deg_c);  nvoxs = np.array(nvoxs)
print(f"Subjects: {len(hi_c)}")
print(f"HiRes perilesional contrast:    {hi_c.mean():+.3f} ± {hi_c.std():.3f}")
print(f"Degraded perilesional contrast: {deg_c.mean():+.3f} ± {deg_c.std():.3f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5), gridspec_kw={"wspace": 0.36})

sc  = ax1.scatter(hi_c, deg_c, c=np.log10(nvoxs), cmap="plasma",
                  s=30, alpha=0.75, edgecolors="none", zorder=3)
lim = max(abs(hi_c).max(), abs(deg_c).max()) * 1.18
ax1.axhline(0, color="black", lw=1.3, ls="--", zorder=2, label="zero contrast")
ax1.axvline(0, color="black", lw=1.3, ls="--", zorder=2)
ax1.plot([-lim, lim], [-lim, lim], color="gray", lw=0.9, ls=":", label="no change")
ax1.set_xlim(-lim, lim);  ax1.set_ylim(-lim, lim)
ax1.axvspan(-lim, 0, alpha=0.05, color=COLOR_HI, zorder=0)
ax1.axhspan(-lim, 0, alpha=0.05, color=COLOR_DEG, zorder=0)
fig.colorbar(sc, ax=ax1, fraction=0.040, pad=0.02).set_label("log₁₀(lesion voxels)", fontsize=9)
ax1.set_xlabel("HiRes perilesional contrast", fontsize=12)
ax1.set_ylabel("Degraded perilesional contrast", fontsize=12)
ax1.set_title(f"Perilesional contrast: HiRes → {METHOD_NAME}\n(n = {len(hi_c)} subjects)",
              fontsize=11, fontweight="bold")
ax1.legend(fontsize=8.5);  ax1.spines["top"].set_visible(False);  ax1.spines["right"].set_visible(False)

all_vals = np.concatenate([hi_c, deg_c])
bins = np.linspace(all_vals.min() * 1.15, all_vals.max() * 1.15, 42)
ax2.hist(hi_c,  bins=bins, alpha=0.72, color=COLOR_HI,  label=f"HiRes  (μ = {hi_c.mean():+.3f})",
         edgecolor="white", linewidth=0.3)
ax2.hist(deg_c, bins=bins, alpha=0.72, color=COLOR_DEG, label=f"Degraded  (μ = {deg_c.mean():+.3f})",
         edgecolor="white", linewidth=0.3)
ax2.axvline(0,           color="black",   lw=1.8, ls="--", zorder=5)
ax2.axvline(hi_c.mean(), color=COLOR_HI,  lw=2.2, ls="-",  zorder=4)
ax2.axvline(deg_c.mean(),color=COLOR_DEG, lw=2.2, ls="-",  zorder=4)
ax2.axvspan(all_vals.min() * 1.15, 0, alpha=0.07, color=COLOR_HI, zorder=0)
ax2.text(0.02, 0.96, "← darker than\n   surrounding tissue",
         transform=ax2.transAxes, va="top", fontsize=9, color=COLOR_HI)
ax2.set_xlabel("Lesion mean − perilesional mean  (a.u.)", fontsize=11)
ax2.set_ylabel("Number of subjects", fontsize=11)
ax2.set_title(f"Effect of {METHOD_NAME}\non lesion perilesional contrast", fontsize=11, fontweight="bold")
ax2.legend(fontsize=9.5);  ax2.spines["top"].set_visible(False);  ax2.spines["right"].set_visible(False)

fig.suptitle("Perilesional contrast attenuation  |  Perilesional ring ≈ 5 mm",
             fontsize=12, fontweight="bold", y=1.02)
plt.savefig(SAVE_PATH, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {SAVE_PATH}")